# --- 1. Setup ---

In [1]:
!pip install openai langchain faiss-cpu tiktoken

In [2]:
from openai import OpenAI
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
import faiss
import numpy as np

client = OpenAI()

# --- 2. Input Text ---

In [3]:
long_text = """
Partnering to bring new advanced AI capabilities to enterprises worldwide
OpenAI and Amazon are jointly developing a Stateful Runtime Environment powered by OpenAI’s models, which will be available through Amazon Bedrock.
Stateful developer environments are the next generation of how frontier models will be used, seamlessly enabling models to access elements like compute, memory, and identity. A Stateful Runtime Environment allows developers to keep context, remember prior work, work across software tools and data sources, and access compute. They're designed to handle ongoing projects and workflows.
These stateful developer environments will be trained to run optimally on AWS's infrastructure and integrated with Amazon Bedrock AgentCore and infrastructure services so customers’ AI applications and agents run cohesively with the rest of their infrastructure applications running in AWS. The Stateful Runtime Environment is expected to launch in the next few months. Contact AWS to learn how it can accelerate your AI journey.
Bringing OpenAI’s most advanced enterprise platform to AWS customers
AWS will serve as the exclusive third-party cloud distribution provider for OpenAI Frontier, expanding access to OpenAI’s most advanced enterprise platform as demand for AI deployment accelerates across industries.
Frontier enables organizations to build, deploy, and manage teams of AI agents that operate across real business systems with shared context, built-in governance, and enterprise-grade security, without managing underlying infrastructure. As companies move from experimentation to production AI, Frontier makes it straightforward to integrate powerful AI into existing workflows quickly, securely, and at global scale.
OpenAI to use Trainium compute to power growing Amazon customer demand
OpenAI and AWS are expanding their existing $38 billion multi-year agreement by $100 billion over 8 years. The expansion includes OpenAI committing to consume approximately 2 gigawatts of Trainium capacity through AWS infrastructure, which will support demand for Stateful Runtime, Frontier, and other advanced workloads. This agreement lowers the cost and improves the efficiency of producing intelligence at scale.
Under this structure, OpenAI secures long-term capacity while working with AWS to deploy purpose-built silicon alongside its broader compute ecosystem, enabling enterprises to consume intelligence on demand without managing underlying infrastructure.
This commitment spans both Trainium3 and next-generation Trainium4 chips and will power a broad range of advanced AI workloads. Trainium4, expected to begin delivery in 2027, will provide another major performance gain, including significantly higher FP4 compute performance, expanded memory bandwidth, and increased high-bandwidth memory capacity to support increasingly capable AI systems at scale.
Custom models available to power Amazon’s customer-facing applications
OpenAI and Amazon will collaborate to develop customized models available to Amazon developers to power Amazon’s customer-facing applications. Amazon teams will be able to tailor OpenAI models for use across AI products and agents that serve customers directly. These capabilities will complement the models already available to Amazon developers, including Amazon’s Nova family, offering another tool for teams to build and deliver at scale.
“OpenAI and Amazon share a belief that AI should show up in ways that are practical and genuinely useful for people,” said Sam Altman, co-founder and CEO of OpenAI. ”Combining OpenAI’s models with Amazon’s infrastructure and global reach helps us put powerful AI into the hands of businesses and users at real scale.”
“We have lots of developers and companies eager to run services powered by OpenAI models on AWS, and our unique collaboration with OpenAI to provide stateful runtime environments will change what’s possible for customers building AI apps and agents,” said Andy Jassy, President and CEO of Amazon. “We continue to be impressed with what OpenAI is building, and we're excited not only about their choosing to go big on our custom AI silicon (Trainium), but also our opportunity to invest in the company and partnership over the long-term.”
About OpenAI
OpenAI is an AI research and deployment company. Our mission is to ensure that artificial general intelligence benefits all of humanity.
About Amazon
Amazon is guided by four principles: customer obsession rather than competitor focus, passion for invention, commitment to operational excellence, and long-term thinking. Amazon strives to be Earth’s Most Customer-Centric Company, Earth’s Best Employer, and Earth’s Safest Place to Work. Customer reviews, 1-Click shopping, personalized recommendations, Prime, Fulfillment by Amazon, AWS, Kindle Direct Publishing, Kindle, Career Choice, Fire tablets, Fire TV, Amazon Echo, Alexa, Just Walk Out technology, Amazon Studios, and The Climate Pledge are some of the things pioneered by Amazon. 
"""

# --- 3. Chunking Strategies ---
# (a) Recursive Character Splitter (preferred: natural boundaries)

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)
chunks_recursive = recursive_splitter.split_text(long_text)

# (b) Simple Character Splitter (fixed size chunks)
Pros: Simple, customizable.
Cons: May cut sentences mid-way.

In [5]:
char_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks_char = char_splitter.split_text(long_text)

# (c) Token-based Splitter (uses tiktoken tokenizer)
token_splitter = TokenTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks_token = token_splitter.split_text(long_text)

print(f"Recursive chunks: {len(chunks_recursive)}")
print(f"Character chunks: {len(chunks_char)}")
print(f"Token chunks: {len(chunks_token)}")

Recursive chunks: 15
Character chunks: 1
Token chunks: 3


# --- 4. Embeddings ---
Generate embeddings for each chunk:
Store embeddings in a vector database (e.g., Pinecone, FAISS).
Retrieve relevant chunks for queries → feed into GPT for summarization/answers.

In [6]:
def embed_text(texts, model="text-embedding-3-small"):
    vectors = []
    for t in texts:
        response = client.embeddings.create(
            model=model,
            input=t
        )
        vectors.append(response.data[0].embedding)
    return np.array(vectors)

# Choose your chunking strategy here:

In [7]:
chunks = chunks_recursive   # or chunks_char / chunks_token
embeddings = embed_text(chunks)

# --- 5. Vector Store (FAISS) ---

In [8]:
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# --- 6. Retrieval Function ---

In [9]:
def retrieve(query, k=3):
    q_embed = embed_text([query])
    distances, indices = index.search(q_embed, k)
    results = [chunks[i] for i in indices[0]]
    return results

# --- 7. Example Query ---

In [10]:
query = "What does this document say about holiday inventory spikes?"
results = retrieve(query)

print("Top relevant chunks:")
for r in results:
    print("----")
    print(r)

Top relevant chunks:
----
This commitment spans both Trainium3 and next-generation Trainium4 chips and will power a broad range of advanced AI workloads. Trainium4, expected to begin delivery in 2027, will provide another major performance gain, including significantly higher FP4 compute performance, expanded memory bandwidth, and increased high-bandwidth memory capacity to support increasingly capable AI systems at scale.
Custom models available to power Amazon’s customer-facing applications
----
OpenAI and AWS are expanding their existing $38 billion multi-year agreement by $100 billion over 8 years. The expansion includes OpenAI committing to consume approximately 2 gigawatts of Trainium capacity through AWS infrastructure, which will support demand for Stateful Runtime, Frontier, and other advanced workloads. This agreement lowers the cost and improves the efficiency of producing intelligence at scale.
----
Under this structure, OpenAI secures long-term capacity while working with 

# --- 8. Optional: Feed retrieved chunks into GPT ---

In [11]:
context = "\n\n".join(results)
completion = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Answer based on context:\n{context}\n\nQuestion: {query}"}
    ]
)

print("\nGPT Answer:\n", completion.choices[0].message.content)


GPT Answer:
 The document provided does not mention or discuss holiday inventory spikes. It focuses on the expansion of the agreement between OpenAI and AWS, the details about new chip technologies, and the implications for AI workload support and infrastructure.
